## Dataset 2 — HAL.science FR (multi-sauts)
100 notices scientifiques — objectif **≥ 40 000 mots** (API détaillée + page HTML)

## 0. Montage Google Drive

In [ ]:
# Montage du Drive et définition du chemin de base du projet
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

## 1. Installation des dépendances

In [ ]:
!pip install -q wikipedia-api feedparser requests beautifulsoup4 pypdf

## 2. Imports et configuration

In [ ]:
import os, json, time, datetime, re
import wikipediaapi, feedparser
import requests
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from collections import Counter

RAW_PATH = os.path.join(BASE_PATH, 'data', 'raw')
os.makedirs(RAW_PATH, exist_ok=True)

WIKI_PATH        = os.path.join(RAW_PATH, 'wikipedia_technique.json')
HAL_PATH         = os.path.join(RAW_PATH, 'hal.json')
LEMONDE_PATH     = os.path.join(RAW_PATH, 'lemonde.json')
LEGAL_DIR        = os.path.join(RAW_PATH, 'legal')
CODE_ROUTE_PATH  = os.path.join(RAW_PATH, 'code_route.json')
os.makedirs(LEGAL_DIR, exist_ok=True)
code_route_articles = []  # rempli par la cellule « PDF code de la route » si un PDF est présent
print(f"Répertoire de sortie : {RAW_PATH}")
print(f"  PDF juridique    : déposez-le dans {LEGAL_DIR} puis exécutez la cellule dédiée.")

## Dataset 1 — Wikipedia FR (technique)
~150+ entrées thématiques IA / ML (titres d'articles à récupérer via l'API).


In [ ]:
# ~150+ topics Wikipedia FR pour scraping IA/ML
WIKI_TOPICS = [
    # Réseaux de neurones
    "Réseau de neurones artificiel", "Réseau de neurones récurrent",
    "Réseau de neurones convolutif", "Perceptron", "Rétropropagation du gradient",
    "Réseau résiduel", "Réseau de neurones profond", "Réseau de neurones à capsules",
    "Réseau de neurones boltzmann", "Machine de Boltzmann restreinte",
    "Réseau de neurones récurrent bidirectionnel", "Réseau de Hopfield",
    "Réseau de neurones à graphe", "Réseau de neurones spiking",
    "Autoencoder", "Autoencoder variationnel",

    # Transformeur & attention
    "Transformeur (apprentissage automatique)", "Mécanisme d'attention",
    "Attention multi-têtes", "Codage positionnel", "Vision Transformer",
    "BERT (modèle de langage)", "GPT", "ChatGPT", "GPT-4", "LLaMA",

    # Apprentissage automatique — fondements
    "Apprentissage automatique", "Apprentissage profond",
    "Apprentissage supervisé", "Apprentissage non supervisé",
    "Apprentissage semi-supervisé", "Apprentissage par transfert",
    "Surajustement", "Régularisation (mathématiques)",
    "Forêt d'arbres décisionnels", "Machine à vecteurs de support",
    "Régression logistique", "Descente de gradient stochastique",
    "Descente de gradient", "Optimisation convexe",
    "Algorithme de rétropropagation", "Régression linéaire",
    "Arbre de décision", "Boosting", "Gradient boosting",
    "XGBoost", "Bagging", "K plus proches voisins",
    "Analyse en composantes principales", "Analyse discriminante linéaire",
    "Clustering", "K-moyennes", "Algorithme espérance-maximisation",
    "Modèle de mélange gaussien", "DBSCAN", "Carte auto-organisatrice",
    "Réduction de dimensionnalité", "t-SNE", "UMAP",

    # NLP
    "Traitement automatique des langues", "Traduction automatique",
    "Analyse de sentiments", "Résumé automatique",
    "Reconnaissance d'entités nommées", "Question-réponse automatique",
    "Génération de texte", "Lemmatisation", "Tokenisation",
    "Racinisation", "Analyse syntaxique", "Désambiguïsation lexicale",
    "Modèle de langue", "N-gramme", "TF-IDF",
    "Plongement de mots", "Word2vec", "FastText", "GloVe",
    "Étiquetage morpho-syntaxique", "Coréférence", "Chatbot",
    "Compréhension du langage naturel", "Génération de langage naturel",
    "Traitement de la parole",

    # Apprentissage par renforcement
    "Apprentissage par renforcement",
    "Apprentissage par renforcement à partir de retours humains",
    "Q-learning", "Processus de décision markovien",
    "Politique (apprentissage par renforcement)", "Récompense (apprentissage par renforcement)",
    "Algorithme acteur-critique", "Optimisation de politique proximale",
    "Apprentissage par renforcement profond", "AlphaGo", "AlphaZero",
    "Apprentissage par imitation",

    # GAN & Génératif
    "Réseau antagoniste génératif", "Stable Diffusion", "DALL-E",
    "Modèle de diffusion", "Réseau antagoniste génératif conditionnel",
    "StyleGAN", "Flux normalisant", "Modèle génératif",
    "Échantillonnage par rejet",

    # Modèles de langage & IA générative
    "Mistral AI", "Claude (assistant)", "Génération augmentée par récupération",
    "Intelligence artificielle générative", "Ingénierie des prompts",
    "Apprentissage zéro-shot", "Apprentissage few-shot", "Ajustement fin",
    "Quantification de modèles", "Distillation de modèle",
    "Élagage (apprentissage automatique)", "Compression de modèle",
    "LoRA (apprentissage automatique)", "RLHF",

    # IA générale & éthique
    "Intelligence artificielle", "Intelligence artificielle générale",
    "Hallucination (intelligence artificielle)", "Biais algorithmique",
    "Éthique de l'intelligence artificielle", "Alignement de l'IA",
    "Réglementation de l'intelligence artificielle",
    "Biais dans les données", "Explicabilité des modèles",
    "Confidentialité différentielle", "Apprentissage fédéré",
    "Sécurité de l'intelligence artificielle", "IA responsable",
    "Test de Turing", "Intelligence artificielle forte",

    # Vision par ordinateur
    "Vision par ordinateur", "Reconnaissance de formes",
    "Détection d'objets", "Segmentation sémantique",
    "Segmentation d'instance", "Reconnaissance faciale",
    "Traitement d'images", "Augmentation de données",
    "Correspondance de caractéristiques", "Estimation de pose",
    "Réseau de neurones convolutif siamois", "OCR",

    # Organisations
    "Hugging Face", "OpenAI", "Anthropic", "DeepMind", "Meta AI",
    "Google Brain", "EleutherAI", "Stability AI",

    # Techniques avancées
    "Normalisation par lots", "Encodeur-décodeur",
    "Mémoire longue à court terme", "Unité récurrente à porte",
    "Intégration de mots", "Inférence bayésienne",
    "Réseau bayésien", "Apprentissage actif",
    "Apprentissage multi-tâches", "Apprentissage de métrique",
    "Apprentissage contrastif", "Auto-supervision",
    "Réseaux siamois", "Apprentissage curriculaire",
    "Optimisation hyper-paramètres", "Recherche d'architecture neuronale",
    "Dropout (réseau de neurones)", "Fonction d'activation",
    "ReLU", "Softmax", "Batch normalization",

    # Infrastructure ML
    "Unité de traitement graphique", "TensorFlow", "PyTorch",
    "Keras", "Scikit-learn", "Apache Spark",
    "MLflow", "Kubeflow", "DataOps", "MLOps",
    "Base de données vectorielle", "Recherche sémantique",
    "FAISS", "Traitement en flux",

    # Évaluation
    "Validation croisée", "Courbe ROC", "Score F1",
    "Matrice de confusion", "Précision et rappel",
    "Perplexité (linguistique)", "BLEU (métrique)",
    "Score ROUGE", "Erreur quadratique moyenne",

    # Domaines d'application
    "Diagnostic médical assisté par ordinateur", "Conduite autonome",
    "Robotique", "Finance algorithmique", "Reconnaissance vocale",
    "Bioinformatique", "Recherche d'information",
    "Fouille de textes", "Détection d'anomalies",
    "Prévision de séries temporelles", "Système de recommandation",
    "Traitement du signal", "Informatique cognitive",
    "Système expert", "Jeu vidéo et intelligence artificielle",
    "Traduction automatique neuronale", "Synthèse vocale",
    "Reconnaissance de la parole", "Veille technologique",
    "Médecine personnalisée", "Astronomie et intelligence artificielle",
    "Agriculture de précision", "Détection de fraude",

    # Mathématiques sous-jacentes
    "Algèbre linéaire", "Calcul tensoriel", "Probabilité",
    "Statistiques bayésiennes", "Entropie (théorie de l'information)",
    "Divergence de Kullback-Leibler", "Théorème de Bayes",
    "Optimisation stochastique", "Théorie de l'information",
    "Calcul différentiel", "Fonction de perte",
]
TARGET_WIKI = 100
print(f"Topics disponibles : {len(WIKI_TOPICS)}")
print(f"Cible              : {TARGET_WIKI} articles (dataset_type='technique') — Option B")


In [ ]:
wiki = wikipediaapi.Wikipedia(
    language='fr',
    user_agent='LLM-Integration-Study/1.0 (research@example.com)'
)
wikipedia_articles = []

for topic in tqdm(WIKI_TOPICS, desc="Wikipedia FR [technique]"):
    if len(wikipedia_articles) >= TARGET_WIKI:
        break
    try:
        page = wiki.page(topic)
        if not page.exists():
            print(f"  [SKIP] Inexistante : {topic}")
            continue
        content = page.text
        if len(content) < 300:
            print(f"  [SKIP] Trop court  : {topic}")
            continue
        wikipedia_articles.append({
            "id":           f"wiki_{len(wikipedia_articles):03d}",
            "title":        page.title,
            "content":      content[:8000],
            "source":       "wikipedia_fr",
            "langue":       "fr",
            "date":         datetime.date.today().isoformat(),
            "url":          page.fullurl,
            "dataset_type": "technique",
        })
        time.sleep(0.3)
    except Exception as e:
        print(f"  [ERROR] {topic} : {e}")

total_words_wiki = sum(len(a['content'].split()) for a in wikipedia_articles)
print(f"\nWikipedia collectés : {len(wikipedia_articles)} / {TARGET_WIKI}  (~{total_words_wiki:,} mots)")

## 4. Scraping HAL.science — 100 articles (multi-sauts), cible **≥ 40 000 mots**

**Temporalité** : collecte par **trois quotas** sur `submittedDateY_i` (même logique que `recency_category` dans le notebook 02 : **année ≤ 2021** fondamental, **2022–2023** intermédiaire, **≥ 2024** récent). Si une strate manque de notices, une passe **complément** élargit la fenêtre. Wikipedia / Le Monde restent surtout « récents » (date de fetch ou d’article) ; la **dispersion temporelle** du corpus Option B repose surtout sur **HAL**.

In [ ]:
# ── HAL.science — archive ouverte française ───────────────────────────────────
# API publique, aucune clé requise, filtrée langue=fr + **fenêtres d’année par strate** (voir HAL_STRATA).
# Option B : 100 papiers → 500 Q&A — objectif corpus ≥ 40 000 mots (≈400 mots/doc en moyenne)

import statistics as _stats
from collections import Counter as _Counter

HAL_API    = 'https://api.archives-ouvertes.fr/search/'
UA_HAL     = 'LLM-Integration-Study/2.1 (research@example.com)'
TARGET_HAL = 100
MIN_WORDS  = 60     # seuil initial (allégé : détail API + page HTML combleront)
HAL_MIN_TOTAL_WORDS = 40000  # objectif global sur les 100 notices

HAL_QUERIES = [
    "apprentissage automatique transformers",
    "traitement automatique langage naturel",
    "grands modèles langage français",
    "apprentissage profond réseaux neurones",
    "génération texte intelligence artificielle",
    "modèles génératifs diffusion",
    "inférence optimisation modèles",
    "représentations vectorielles sémantiques",
    "classification texte analyse sentiment",
    "traduction automatique neuronale",
    "résumé automatique extraction",
    "vision par ordinateur détection objets",
    "apprentissage par renforcement agent",
    "évaluation modèles langage biais",
    "alignement éthique intelligence artificielle",
    "large language model français",
    "modèle de langage préalablement entraîné",
    "apprentissage par transfert deep learning",
]

# Clés Solr HAL utiles pour maximiser le texte (résumés, vulgarisation, etc.)
HAL_TEXT_KEYS = (
    "abstract_s", "popularScienceAbstract_s", "reviewAbstract_s",
    "description_s", "structResume_s", "label_biblio_s",
)


def _dedupe_blocks(blocks):
    seen, out = set(), []
    for b in blocks:
        b = re.sub(r"\s+", " ", (b or "").strip())
        if len(b.split()) < 15:
            continue
        h = b[:240]
        if h not in seen:
            seen.add(h)
            out.append(b)
    return out


def hal_api_detail(hal_id):
    """Récupère toutes les métadonnées textuelles disponibles pour un halId."""
    try:
        r = requests.get(HAL_API, headers={"User-Agent": UA_HAL}, params={
            "q":    f"halId_s:{hal_id}",
            "rows": 1,
            "wt":   "json",
            "fl":   "*",
        }, timeout=15)
        r.raise_for_status()
        docs = r.json().get("response", {}).get("docs", [])
        return docs[0] if docs else {}
    except Exception:
        return {}


def compose_hal_text(doc, title_s, authors_s, kws_s):
    """Assemble un texte long à partir des champs API (toutes sources FR possibles)."""
    parts = []
    if title_s:
        parts.append("Titre : " + title_s.strip())
    if authors_s:
        parts.append("Auteurs : " + authors_s)
    if kws_s:
        parts.append("Mots-clés : " + kws_s)

    for key in HAL_TEXT_KEYS:
        v = doc.get(key)
        if v is None:
            continue
        if isinstance(v, list):
            for item in v:
                s = str(item).strip()
                if len(s.split()) >= 20:
                    parts.append(s)
        elif isinstance(v, str) and len(v.split()) >= 20:
            parts.append(v.strip())

    text = "\n\n".join(_dedupe_blocks(parts))
    return text


def get_hal_page_text(hal_id, max_chars=18000):
    """Extrait un maximum de texte français depuis la page publique hal.science."""
    try:
        url = f"https://hal.science/{hal_id}"
        r = requests.get(url, headers={"User-Agent": UA_HAL}, timeout=14)
        if r.status_code != 200:
            return ""
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        chunks = []
        for sel in [
            "div.abstract", "section.abstract", ".abstract-content",
            "div[class*='abstract']", "#abstract", "[property='abstract']",
        ]:
            for el in soup.select(sel):
                t = el.get_text(separator=" ", strip=True)
                if len(t.split()) > 40:
                    chunks.append(t)

        main = soup.find("article") or soup.find("main") or soup.find(
            "div", class_=re.compile(r"content|metadata-body", re.I))
        if main:
            for nav in main.find_all(["nav", "footer", "aside"]):
                nav.decompose()
            paras = []
            for p in main.find_all("p"):
                t = p.get_text(separator=" ", strip=True)
                if len(t) > 55 and not t.lower().startswith("copyright"):
                    paras.append(t)
            if paras:
                body = " ".join(paras[:80])
                chunks.append(body)

        blob = "\n\n".join(_dedupe_blocks(chunks))
        return blob[:max_chars]
    except Exception:
        return ""


# ── Scraping principal via l'API HAL (quotas temporels) ───────────────────
# Même découpage annuel que infer_recency (02) : ≤2021 / 2022-2023 / ≥2024

def _hal_fq_date(year_fq_solr):
    return ["language_s:fr", year_fq_solr, "(docType_s:ART OR docType_s:COMM OR docType_s:PREPRINT)"]

_hal_targets = [TARGET_HAL // 3 + (1 if i < TARGET_HAL % 3 else 0) for i in range(3)]
HAL_STRATA = [
    ("fondamental",     "submittedDateY_i:[1980 TO 2021]", _hal_targets[0]),
    ("intermédiaire",   "submittedDateY_i:[2022 TO 2023]", _hal_targets[1]),
    ("récent",          "submittedDateY_i:[2024 TO 2030]", _hal_targets[2]),
]

hal_papers = []
seen_ids   = set()

def _hal_collect_one_batch(docs, stratum_label):
    """Ajoute jusqu'à concurrence des docs valides ; retourne nb ajoutés."""
    added = 0
    for doc in docs:
        hal_id = doc.get("halId_s", "")
        if not hal_id or hal_id in seen_ids:
            continue
        title_l = doc.get("title_s") or [""]
        title = title_l[0] if isinstance(title_l, list) else title_l
        date_raw = (doc.get("submittedDate_s") or "")[:10]
        url = doc.get("uri_s", f"https://hal.science/{hal_id}")
        authors = ", ".join((doc.get("authFullName_s") or [])[:5])
        kws = ", ".join((doc.get("keyword_s") or [])[:10])
        detail = hal_api_detail(hal_id)
        merged = {**doc, **detail}
        content = compose_hal_text(merged, title, authors, kws)
        if len(content.split()) < MIN_WORDS:
            continue
        seen_ids.add(hal_id)
        hal_papers.append({
            "id":                 f"hal_{len(hal_papers):03d}",
            "title":              title,
            "content":            content,
            "source":             "hal",
            "langue":             "fr",
            "date":               date_raw,
            "url":                url,
            "hal_id":             hal_id,
            "authors":            authors,
            "dataset_type":       "multisauts",
            "hal_scrape_stratum": stratum_label,
        })
        added += 1
        time.sleep(0.25)
    return added

for stratum_label, year_fq, goal in HAL_STRATA:
    n0 = len(hal_papers)
    for query in tqdm(HAL_QUERIES, desc=f"HAL [{stratum_label}]"):
        if len(hal_papers) - n0 >= goal:
            break
        try:
            resp = requests.get(HAL_API, headers={"User-Agent": UA_HAL}, params={
                "q":    query,
                "fq":   _hal_fq_date(year_fq),
                "fl":   "halId_s,title_s,abstract_s,uri_s,submittedDate_s,authFullName_s,keyword_s,domainAllCode_s",
                "rows": 50,
                "wt":   "json",
                "sort": "submittedDate_s desc",
            }, timeout=15)
            resp.raise_for_status()
            docs = resp.json().get("response", {}).get("docs", [])
            time.sleep(0.35)
            for doc in docs:
                if len(hal_papers) - n0 >= goal:
                    break
                _hal_collect_one_batch([doc], stratum_label)
        except Exception as e:
            print(f"  [ERROR] [{stratum_label}] '{query[:50]}' : {e}")
    got = len(hal_papers) - n0
    print(f"  Strate '{stratum_label}' : {got} / {goal} notices")

if len(hal_papers) < TARGET_HAL:
    miss = TARGET_HAL - len(hal_papers)
    print(f"  [INFO] Complément HAL (fenêtre 1980–2030, {miss} notices manquantes)...")
    for query in tqdm(HAL_QUERIES, desc="HAL [complément]"):
        if len(hal_papers) >= TARGET_HAL:
            break
        try:
            resp = requests.get(HAL_API, headers={"User-Agent": UA_HAL}, params={
                "q":    query,
                "fq":   _hal_fq_date("submittedDateY_i:[1980 TO 2030]"),
                "fl":   "halId_s,title_s,abstract_s,uri_s,submittedDate_s,authFullName_s,keyword_s,domainAllCode_s",
                "rows": 60,
                "wt":   "json",
                "sort": "submittedDate_s desc",
            }, timeout=15)
            resp.raise_for_status()
            docs = resp.json().get("response", {}).get("docs", [])
            time.sleep(0.35)
            for doc in docs:
                if len(hal_papers) >= TARGET_HAL:
                    break
                if doc.get("halId_s") in seen_ids:
                    continue
                _hal_collect_one_batch([doc], "complément")
        except Exception as e:
            print(f"  [ERROR] [complément] '{query[:50]}' : {e}")

if hal_papers:
    print("  Répartition strates (scraping) :", dict(_Counter(p.get("hal_scrape_stratum", "?") for p in hal_papers)))

print(f"HAL collectés (API détaillée) : {len(hal_papers)} / {TARGET_HAL}")

# ── Enrichissement page HTML (souvent plus long que la seule notice Solr) ───
print(f"\nEnrichissement page hal.science ({len(hal_papers)} papiers)...")
enriched = 0
for paper in tqdm(hal_papers, desc="HAL page enrichment"):
    page_txt = get_hal_page_text(paper["hal_id"])
    if page_txt:
        merged_blocks = _dedupe_blocks([paper["content"], page_txt])
        new_c = "\n\n".join(merged_blocks)
        if len(new_c.split()) > len(paper["content"].split()):
            paper["content"] = new_c[:22000]
            enriched += 1
    time.sleep(0.25)

words_hal = [len(p["content"].split()) for p in hal_papers]
total_words_hal = sum(words_hal)
print(f"  Papiers enrichis (page > API) : {enriched} / {len(hal_papers)}")
if words_hal:
    print(f"  Mots/papier (médiane) : {_stats.median(words_hal):.0f}")
    print(f"  Mots/papier (moyenne) : {_stats.mean(words_hal):.0f}")
print(f"\nHAL total : {len(hal_papers)} papiers  ~{total_words_hal:,} mots")

if total_words_hal < HAL_MIN_TOTAL_WORDS and hal_papers:
    print(f"  [INFO] Sous l'objectif {HAL_MIN_TOTAL_WORDS:,} mots — 2e passe HTML (extraits longs)...")
    for paper in tqdm(hal_papers, desc="HAL 2e passe"):
        extra = get_hal_page_text(paper["hal_id"], max_chars=32000)
        if extra:
            m = "\n\n".join(_dedupe_blocks([paper["content"], extra]))
            paper["content"] = m[:28000]
        time.sleep(0.2)
    words_hal = [len(p["content"].split()) for p in hal_papers]
    total_words_hal = sum(words_hal)
    print(f"  Après 2e passe : ~{total_words_hal:,} mots  (moy. {_stats.mean(words_hal):.0f}/doc)")

dates_hal = sorted(p["date"] for p in hal_papers if p["date"])
if dates_hal:
    print(f"  Période : {dates_hal[0]} → {dates_hal[-1]}")

if total_words_hal < HAL_MIN_TOTAL_WORDS:
    print(f"  [WARN] Objectif {HAL_MIN_TOTAL_WORDS:,} mots non atteint ({total_words_hal:,}) — "
          f"ajoutez des requêtes dans HAL_QUERIES ou augmentez TARGET_HAL.")

if len(hal_papers) < TARGET_HAL:
    print(f"  [WARN] Cible notices : {len(hal_papers)}/{TARGET_HAL} — ajoutez des requêtes dans HAL_QUERIES")


## Dataset 3 — Le Monde RSS (temporel)
Articles d’actualité technologie & sciences via flux RSS (**très récents** par nature). Pour une vraie profondeur temporelle « ancien / intermédiaire / récent », le scraping **HAL** ci-dessus porte l’essentiel du signal ; ce bloc sert surtout au **domaine presse** et à l’actualité.

In [ ]:
from bs4 import BeautifulSoup
import time as _time
import requests as _requests

# ── Constantes ────────────────────────────────────────────────────────
TARGET_LEMONDE = 100
MIN_WORDS      = 150   # Seuil minimal pour qu'un article soit conservé
UA = 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36'

# ── Scraper générique ─────────────────────────────────────────────────
def scrape_article(url, selectors=None):
    """Scrape le contenu texte d'un article à partir de son URL.
    selectors = liste de (tag, class_fragment) à essayer dans l'ordre."""
    selectors = selectors or [
        ('p', 'article__paragraph'),   # Le Monde
        ('p', 'a-paragraph'),          # France Info
        ('p', 'article-text'),
    ]
    try:
        resp = _requests.get(url, headers={'User-Agent': UA}, timeout=12)
        if resp.status_code != 200:
            return ''
        soup = BeautifulSoup(resp.text, 'html.parser')
        for tag, cls_frag in selectors:
            paras = soup.find_all(tag, class_=lambda c: c and cls_frag in c)
            if paras:
                text = ' '.join(p.get_text().strip() for p in paras)
                text = re.sub(r'\s+', ' ', text).strip()
                if len(text.split()) >= MIN_WORDS:
                    return text[:6000]
        # Fallback générique : balise <article>
        art = soup.find('article')
        if art:
            text = ' '.join(p.get_text().strip()
                            for p in art.find_all('p') if len(p.get_text().strip()) > 30)
            text = re.sub(r'\s+', ' ', text).strip()
            return text[:6000] if len(text.split()) >= MIN_WORDS else ''
        return ''
    except Exception as e:
        return ''
    finally:
        _time.sleep(0.6)   # Respectueux du serveur

# ── Flux RSS à essayer ────────────────────────────────────────────────
LEMONDE_FEEDS = [
    ("le-monde/une",           "https://www.lemonde.fr/rss/une.xml"),
    ("le-monde/sciences",      "https://www.lemonde.fr/sciences/rss_full.xml"),
    ("le-monde/ia",            "https://www.lemonde.fr/intelligence-artificielle/rss_full.xml"),
    ("le-monde/pixels",        "https://www.lemonde.fr/pixels/rss_full.xml"),
    ("le-monde/economie",      "https://www.lemonde.fr/economie/rss_full.xml"),
    ("le-monde/international", "https://www.lemonde.fr/international/rss_full.xml"),
    ("le-monde/societe",       "https://www.lemonde.fr/societe/rss_full.xml"),
    ("le-monde/planete",       "https://www.lemonde.fr/planete/rss_full.xml"),
    ("le-monde/culture",       "https://www.lemonde.fr/culture/rss_full.xml"),
]

# France Info — fallback 100 % open access
FRANCEINFO_FEEDS = [
    ("france-info/titres",    "https://www.francetvinfo.fr/titres.rss"),
    ("france-info/sciences",  "https://www.francetvinfo.fr/sciences.rss"),
    ("france-info/economie",  "https://www.francetvinfo.fr/economie.rss"),
    ("france-info/societe",   "https://www.francetvinfo.fr/societe.rss"),
    ("france-info/politique", "https://www.francetvinfo.fr/politique.rss"),
    ("france-info/monde",     "https://www.francetvinfo.fr/monde.rss"),
]

# ── Extraction RSS → collecte des articles ────────────────────────────
def collect_from_feeds(feeds, target, existing_urls=None):
    """Parcourt des flux RSS, scrape chaque article, ne garde que >= MIN_WORDS."""
    articles = []
    seen = set(existing_urls or [])
    for feed_name, feed_url in feeds:
        if len(articles) >= target:
            break
        try:
            resp = _requests.get(feed_url, headers={'User-Agent': UA}, timeout=15)
            if resp.status_code != 200:
                print(f"  [SKIP] {feed_name} → HTTP {resp.status_code}")
                continue
            feed = feedparser.parse(resp.content)
            print(f"  Flux '{feed_name}' : {len(feed.entries)} entrées RSS")
            for entry in feed.entries:
                if len(articles) >= target:
                    break
                url = entry.get('link', '')
                if url in seen:
                    continue
                seen.add(url)
                # Scraping de l'article complet
                full_text = scrape_article(url)
                if not full_text:
                    # Fallback : titre + résumé RSS (accepté même si court)
                    rss_summary = entry.get('summary', '')
                    rss_summary = re.sub(r'<[^>]+>', ' ', rss_summary)
                    rss_summary = re.sub(r'\s+', ' ', rss_summary).strip()
                    full_text = (entry.get('title','') + '. ' + rss_summary).strip()
                    if len(full_text.split()) < MIN_WORDS:
                        continue   # Trop court même avec le fallback RSS
                # Date
                pub_date = ''
                if hasattr(entry, 'published_parsed') and entry.published_parsed:
                    try:
                        pub_date = _time.strftime('%Y-%m-%d', entry.published_parsed)
                    except Exception:
                        pass
                if not pub_date:
                    pub_date = str(entry.get('published', ''))[:10]
                source_name = feed_name.split('/')[0]
                articles.append({
                    "id":           f"news_{len(articles):03d}",
                    "title":        entry.get('title', '').strip(),
                    "content":      full_text[:6000],
                    "source":       source_name,
                    "langue":       "fr",
                    "date":         pub_date,
                    "url":          url,
                    "feed":         feed_name,
                    "dataset_type": "temporel",
                })
                words = len(full_text.split())
                print(f"    ✓ {entry.get('title','')[:45]:<45}  {words:>4} mots")
        except Exception as e:
            print(f"  [ERROR] {feed_name} : {e}")
    return articles, seen

# ── Passe 1 : Le Monde ────────────────────────────────────────────────
print("Passe 1 — Le Monde (scraping article complet, filtre >= {MIN_WORDS} mots)...")
lemonde_articles, seen_urls = collect_from_feeds(LEMONDE_FEEDS, TARGET_LEMONDE)
print(f"  Le Monde : {len(lemonde_articles)} / {TARGET_LEMONDE} articles ({MIN_WORDS}+ mots)")

# ── Passe 2 : France Info si quota non atteint ────────────────────────
if len(lemonde_articles) < TARGET_LEMONDE:
    remaining = TARGET_LEMONDE - len(lemonde_articles)
    print(f"\nPasse 2 — France Info (fallback open-access, besoin de {remaining} articles)...")
    fi_articles, _ = collect_from_feeds(FRANCEINFO_FEEDS, remaining, seen_urls)
    # Re-numéroter les IDs
    for i, a in enumerate(fi_articles):
        a['id'] = f"news_{len(lemonde_articles)+i:03d}"
    lemonde_articles.extend(fi_articles)
    print(f"  France Info : {len(fi_articles)} articles ajoutés")

total_words_lemonde = sum(len(a['content'].split()) for a in lemonde_articles)
sources = {}
for a in lemonde_articles:
    sources[a['source']] = sources.get(a['source'], 0) + 1

print(f"\nActualités collectées : {len(lemonde_articles)} / {TARGET_LEMONDE}")
print(f"  Mots total           : {total_words_lemonde:,}  (~{total_words_lemonde/max(1,len(lemonde_articles)):.0f} mots/article)")
print(f"  Sources              : {sources}")
if len(lemonde_articles) < TARGET_LEMONDE:
    print(f"  [WARN] Cible non atteinte : {len(lemonde_articles)}/{TARGET_LEMONDE}")

## Dataset 4 — Code de la route (PDF, optionnel)

**Chemins** : un ou plusieurs `.pdf` dans `data/raw/legal/` **ou** `data/raw/code_route.pdf`. Extraction texte (PyPDF) puis découpe en segments (~1400 mots) → `code_route.json` (`dataset_type` : `juridique`). Sans PDF, un fichier vide `[]` est écrit pour que le notebook 02 charge sans erreur.

In [ ]:
# PDF(s) → code_route.json (corpus juridique pour Q&R + RAG)
from pypdf import PdfReader

def _find_code_route_pdfs():
    out = []
    if os.path.isdir(LEGAL_DIR):
        for fn in sorted(os.listdir(LEGAL_DIR)):
            if fn.lower().endswith(".pdf"):
                out.append(os.path.join(LEGAL_DIR, fn))
    for fn in ("code_route.pdf", "Code_de_la_route.pdf"):
        p = os.path.join(RAW_PATH, fn)
        if os.path.isfile(p):
            out.append(p)
    return sorted(set(out))

def _pdf_to_text(path):
    reader = PdfReader(path)
    parts = []
    for page in reader.pages:
        try:
            parts.append(page.extract_text() or "")
        except Exception:
            parts.append("")
    return re.sub(r"\s+", " ", "\n".join(parts)).strip()

def _split_word_chunks(text, words_per_chunk=1400, max_chunks=40):
    w = text.split()
    if not w:
        return []
    chunks, i = [], 0
    while i < len(w) and len(chunks) < max_chunks:
        piece = w[i : i + words_per_chunk]
        if len(piece) < 120:
            if not chunks:
                chunks.append(" ".join(piece))
            break
        chunks.append(" ".join(piece))
        i += words_per_chunk
    return chunks

code_route_articles.clear()
pdfs = _find_code_route_pdfs()
if not pdfs:
    print("[INFO] Aucun PDF code de la route — code_route.json sera vide [].")
else:
    for pdf_path in pdfs:
        print(f"  Lecture : {pdf_path}")
        try:
            raw_txt = _pdf_to_text(pdf_path)
        except Exception as e:
            print(f"  [ERROR] {e}")
            raw_txt = ""
        if len(raw_txt.split()) < 80:
            print("  [WARN] Texte trop court ou PDF scanné sans couche texte (OCR non inclus).")
            continue
        for j, chunk in enumerate(_split_word_chunks(raw_txt)):
            code_route_articles.append({
                "id":           f"cdr_{len(code_route_articles):03d}",
                "title":        f"Code de la route — extrait {j+1}",
                "content":      chunk[:120000],
                "source":       "code_route_pdf",
                "langue":       "fr",
                "date":         datetime.date.today().isoformat(),
                "url":          f"file:{os.path.basename(pdf_path)}#{j}",
                "dataset_type": "juridique",
            })
try:
    with open(CODE_ROUTE_PATH, "w", encoding="utf-8") as f:
        json.dump(code_route_articles, f, ensure_ascii=False, indent=2)
    print(f"  → {len(code_route_articles)} segments écrits : {CODE_ROUTE_PATH}")
except Exception as e:
    print(f"  [ERROR] Sauvegarde : {e}")


## 5. Sauvegarde des données brutes sur Drive

In [ ]:
# Sauvegarde des datasets sur Google Drive (dont code_route.json, éventuellement déjà écrit)
for data, path, label in [
    (wikipedia_articles, WIKI_PATH,       'Wikipedia technique'),
    (hal_papers,         HAL_PATH,        'HAL multisauts'),
    (lemonde_articles,   LEMONDE_PATH,    'Le Monde temporel'),
    (code_route_articles, CODE_ROUTE_PATH, 'Code route (PDF)'),
]:
    try:
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        words = sum(len(d['content'].split()) for d in data)
        print(f"  {label:<25}: {len(data):>3} docs  ~{words:>7,} mots  {path}")
    except Exception as e:
        print(f"  [ERROR] {label} : {e}")

## 6. Statistiques sur les données brutes

In [ ]:
import statistics

all_docs = wikipedia_articles + hal_papers + lemonde_articles + code_route_articles
all_words = sum(len(d['content'].split()) for d in all_docs)

print("=" * 60)
print("STATISTIQUES — Données brutes (Option B + PDF juridique optionnel)")
print("=" * 60)

for label, docs in [("Wikipedia technique", wikipedia_articles),
                    ("HAL multisauts (FR)",  hal_papers),
                    ("Le Monde temporel",    lemonde_articles),
                    ("Code route (PDF)",     code_route_articles)]:
    if not docs:
        continue
    wc = [len(d['content'].split()) for d in docs]
    print(f"\n  {label} ({len(docs)} docs)")
    print(f"    Mots/doc  : min={min(wc)}  moy={statistics.mean(wc):.0f}  médiane={statistics.median(wc):.0f}  max={max(wc)}")
    print(f"    Total     : {sum(wc):,} mots")

print(f"\n{'─'*45}")
print(f"  TOTAL : {len(all_docs)} docs — {all_words:,} mots")
print(f"{'─'*45}")
target_ok = '✔ ATTEINT' if all_words >= 100000 else f'⚠ {all_words:,} mots'
print(f"  Objectif 100k+ mots : {target_ok}")

## 6. Résumé final

In [ ]:
all_docs  = wikipedia_articles + hal_papers + lemonde_articles + code_route_articles
all_words = sum(len(d['content'].split()) for d in all_docs)

print("=" * 65)
print("RÉSUMÉ — Notebook 01 : datasets collectés (Option B + juridique optionnel)")
print("=" * 65)
print(f"\n{'Source':<25} {'Docs':>5} {'~Mots':>8}   dataset_type")
print("-" * 65)
for label, docs, dtype in [
    ("Wikipedia FR",   wikipedia_articles, "technique"),
    ("HAL.science FR", hal_papers,         "multisauts"),
    ("Le Monde / FI",  lemonde_articles,   "temporel"),
    ("Code route PDF", code_route_articles, "juridique"),
]:
    w = sum(len(d['content'].split()) for d in docs)
    print(f"  {label:<23} {len(docs):>5} {w:>8,}   {dtype}")
print("-" * 65)
print(f"  {'TOTAL':<23} {len(all_docs):>5} {all_words:>8,}")

print("\n  Q&A prévues (5 paires/doc) — Option B 80/20 :")
print(f"    Wikipedia  : {len(wikipedia_articles)*5:>4} paires → 400 train + 100 test (320+80 si juridique actif dans 02)")
print(f"    HAL        : {len(hal_papers)*5:>4} paires → 200 train (100s+100c) + 100 test (50s+50c)")
print(f"    Le Monde   : {len(lemonde_articles)*5:>4} paires → 400 train + 100 test")
_cdr = len(code_route_articles)
if _cdr >= 20:
    print(f"    Code route : 100 paires max (20 segments × 5) → 80 train + 20 test")
elif _cdr > 0:
    print(f"    Code route : {_cdr} segment(s) — il en faut ≥20 pour le quota juridique du notebook 02.")
total_qa  = len(wikipedia_articles)*5 + len(hal_papers)*5 + len(lemonde_articles)*5
print(f"    TOTAL (3 sources) : {total_qa:>4} paires → 1200 train + 300 test ; +juridique si PDF complet.")

print("\n✔ Notebook 01 terminé. Lancez 02_dataset_builder.ipynb.")
print("=" * 65)